In [0]:
from utils.logger import get_logger

CATALOG = "acme_catalog"
GOLD_SCHEMA = "gold"
GOLD = f"{CATALOG}.{GOLD_SCHEMA}"

logger = get_logger("gold_marts")

In [0]:
spark.sql(f"""
    create or replace table {GOLD}.agg_sales_by_month
    comment 'Gold KPI: monthly sales aggregated by division, country and category'
    as
    select
        dd.year, dd.quarter, dd.month, dd.month_name,
        cast(dd.year * 10000 + dd.month * 100 + 1 as int) as date_id,
        dc.division_id, dc.country, dp.category_id,
        cast(sum(fol.revenue) as decimal(14,2)) as total_revenue,
        cast(sum(fol.cost) as decimal(14,2)) as total_cost,
        cast(sum(fol.margin) as decimal(14,2)) as total_margin,
        count(distinct fo.order_id) as order_count,
        count(distinct fo.customer_sk) as distinct_customers
    from {GOLD}.fact_order_lines fol
    join {GOLD}.fact_orders fo on fol.order_id = fo.order_id
    join {GOLD}.dim_date dd on fo.date_id = dd.date_id
    join {GOLD}.dim_customers dc on fo.customer_sk = dc.customer_sk and dc.is_current = true
    join {GOLD}.dim_products dp on fol.product_sk = dp.product_sk and dp.is_current = true
    group by dd.year, dd.quarter, dd.month, dd.month_name, dc.division_id, dc.country, dp.category_id
""")

row_count = spark.table(f"{GOLD}.agg_sales_by_month").count()
logger.info(f"agg_sales_by_month rebuilt: {row_count} rows")

In [0]:
spark.sql(f"""
    create or replace table {GOLD}.agg_customer_growth_by_month
    comment 'Gold KPI: monthly customer growth with new, active and YTD counts'
    as
    with monthly_customers as (
        select
            dd.year, dd.quarter, dd.month, dd.month_name,
            dc.customer_id,
            min(dd.full_date) over (
                partition by dc.customer_id order by dd.full_date
                rows between unbounded preceding and unbounded following
            ) as first_order_date
        from {GOLD}.fact_orders fo
        join {GOLD}.dim_date dd on fo.date_id = dd.date_id
        join {GOLD}.dim_customers dc on fo.customer_sk = dc.customer_sk
    ),
    monthly_agg as (
        select
            year, quarter, month, month_name,
            count(distinct customer_id) as active_customers,
            count(distinct case
                when date_format(first_order_date, 'yyyyMM') = concat(year, lpad(month, 2, '0'))
                then customer_id
            end) as new_customers
        from monthly_customers
        group by year, quarter, month, month_name
    )
    select
        year, quarter, month, month_name, new_customers, active_customers,
        sum(new_customers) over (
            partition by year order by month
            rows between unbounded preceding and current row
        ) as ytd_customers
    from monthly_agg
""")

row_count = spark.table(f"{GOLD}.agg_customer_growth_by_month").count()
logger.info(f"agg_customer_growth_by_month rebuilt: {row_count} rows")

### 1. Analyze sales figures and compare them year-over-year

In [0]:
%sql
with monthly_revenue as (
    select year, month, month_name, sum(total_revenue) as revenue
    from acme_catalog.gold.agg_sales_by_month
    group by year, month, month_name
)
select
    curr.year,
    curr.month,
    curr.month_name,
    curr.revenue as current_year_revenue,
    prev.revenue as last_year_revenue,
    curr.revenue - prev.revenue as yoy_change,
    round((curr.revenue - prev.revenue) / nullif(prev.revenue, 0) * 100, 2) as yoy_pct_change
from monthly_revenue curr
left join monthly_revenue prev
    on curr.month = prev.month and curr.year = prev.year + 1
order by curr.year, curr.month

### 2. Measure and track sales margins

In [0]:
%sql
with order_level_freight as (
    select
        year(order_date) as year,
        month(order_date) as month,
        sum(freight) as total_freight
    from acme_catalog.gold.fact_orders
    group by year(order_date), month(order_date)
),
monthly_gross_margin as (
    select
        year, quarter, month, month_name,
        sum(total_revenue) as total_revenue,
        sum(total_cost) as total_cost,
        sum(total_margin) as gross_margin
    from acme_catalog.gold.agg_sales_by_month
    group by year, quarter, month, month_name
)
select
    m.year,
    m.quarter,
    m.month,
    m.month_name,
    m.total_revenue,
    m.total_cost,
    m.gross_margin,
    round(m.gross_margin / nullif(m.total_revenue, 0) * 100, 2) as gross_margin_pct,
    f.total_freight,
    m.gross_margin - f.total_freight as net_margin,
    round((m.gross_margin - f.total_freight) / nullif(m.total_revenue, 0) * 100, 2) as net_margin_pct
from monthly_gross_margin m
left join order_level_freight f
    on m.year = f.year and m.month = f.month
order by m.year, m.month

### 3. Analyze sales data by various dimensions (Region, Country, Division, Product Line, Product Group, Customer)

In [0]:
%sql

select
    dd.division_name as region,
    dp.category_name as product_line,
    sum(a.total_revenue) as total_revenue,
    sum(a.total_margin) as total_margin
from acme_catalog.gold.agg_sales_by_month a
join acme_catalog.gold.dim_division dd on a.division_id = dd.division_id
join acme_catalog.gold.dim_products dp on a.category_id = dp.category_id and dp.is_current = true
group by dd.division_name, dp.category_name
order by total_revenue desc

### By Country

In [0]:
%sql
select
    country,
    sum(total_revenue) as total_revenue,
    sum(total_margin) as total_margin,
    round(sum(total_margin) / nullif(sum(total_revenue), 0) * 100, 2) as margin_pct,
    sum(order_count) as total_orders
from acme_catalog.gold.agg_sales_by_month
group by country
order by total_revenue desc

### By Customer

In [0]:
%sql
select
    dc.company_name,
    dc.country,
    sum(fol.revenue) as total_revenue,
    sum(fol.margin) as total_margin,
    count(distinct fo.order_id) as order_count
from acme_catalog.gold.fact_order_lines fol
join acme_catalog.gold.fact_orders fo on fol.order_id = fo.order_id
join acme_catalog.gold.dim_customers dc on fo.customer_sk = dc.customer_sk and dc.is_current = true
group by dc.company_name, dc.country
order by total_revenue desc

### By Product Line

In [0]:
%sql
select
    dp.category_name as product_line,
    sum(a.total_revenue) as total_revenue,
    sum(a.total_margin) as total_margin,
    round(sum(a.total_margin) / nullif(sum(a.total_revenue), 0) * 100, 2) as margin_pct,
    sum(a.order_count) as total_orders
from acme_catalog.gold.agg_sales_by_month a
join acme_catalog.gold.dim_products dp on a.category_id = dp.category_id and dp.is_current = true
group by dp.category_name
order by total_revenue desc

### 4. Compute average sales per transaction and per customer

In [0]:
%sql
select
    dd.year,
    dd.month,
    dd.month_name,
    sum(fol.revenue) as total_revenue,
    count(distinct fo.order_id) as total_orders,
    count(distinct fo.customer_sk) as total_customers,
    round(sum(fol.revenue) / nullif(count(distinct fo.order_id), 0), 2) as avg_revenue_per_transaction,
    round(sum(fol.revenue) / nullif(count(distinct fo.customer_sk), 0), 2) as avg_revenue_per_customer
from acme_catalog.gold.fact_order_lines fol
join acme_catalog.gold.fact_orders fo on fol.order_id = fo.order_id
join acme_catalog.gold.dim_date dd on fo.date_id = dd.date_id
group by dd.year, dd.month, dd.month_name
order by dd.year, dd.month

### 5. Customer growth — YTD vs LYTD

In [0]:
%sql
select
    curr.year,
    curr.month,
    curr.month_name,
    curr.new_customers,
    curr.active_customers,
    curr.ytd_customers,
    prev.ytd_customers as lytd_customers,
    curr.ytd_customers - prev.ytd_customers as ytd_vs_lytd_change,
    round((curr.ytd_customers - prev.ytd_customers) / nullif(prev.ytd_customers, 0) * 100, 2) as ytd_vs_lytd_pct
from acme_catalog.gold.agg_customer_growth_by_month curr
left join acme_catalog.gold.agg_customer_growth_by_month prev
    on curr.month = prev.month and curr.year = prev.year + 1
order by curr.year, curr.month